In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)

df = pd.concat(
    [adult.data.features, adult.data.targets],
    axis=1
)

print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
print(df.columns.tolist())

['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


In [4]:
df = df.rename(columns={
    'education-num': 'education_num',
    'marital-status': 'marital_status',
    'capital-gain': 'capital_gain',
    'capital-loss': 'capital_loss',
    'hours-per-week': 'hours_per_week',
    'native-country': 'native_country'
})

print(df.columns.tolist())

['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


In [5]:
csv_path = '/content/hw3_adult.csv'

df.to_csv(csv_path, index=False)

print("CSV saved to:", csv_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

CSV saved to: /content/hw3_adult.csv
Rows: 48842
Columns: 15


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education_num   48842 non-null  int64 
 5   marital_status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital_gain    48842 non-null  int64 
 11  capital_loss    48842 non-null  int64 
 12  hours_per_week  48842 non-null  int64 
 13  native_country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [7]:
df.head(10)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica,<=50K
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States,>50K
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States,>50K


Staging table

In [8]:
staging_sql = """
DROP TABLE IF EXISTS hw3_adult CASCADE;
DROP TABLE IF EXISTS hw3_adult_staging CASCADE;

CREATE TABLE hw3_adult_staging (
    age_raw             TEXT,
    workclass_raw       TEXT,
    fnlwgt_raw          TEXT,
    education_raw       TEXT,
    education_num_raw   TEXT,
    marital_status_raw  TEXT,
    occupation_raw      TEXT,
    relationship_raw    TEXT,
    race_raw            TEXT,
    sex_raw             TEXT,
    capital_gain_raw    TEXT,
    capital_loss_raw    TEXT,
    hours_per_week_raw  TEXT,
    native_country_raw  TEXT,
    income_raw          TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(staging_sql))

print("Staging table created.")

Staging table created.


In [9]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY hw3_adult_staging (
                    age_raw,
                    workclass_raw,
                    fnlwgt_raw,
                    education_raw,
                    education_num_raw,
                    marital_status_raw,
                    occupation_raw,
                    relationship_raw,
                    race_raw,
                    sex_raw,
                    capital_gain_raw,
                    capital_loss_raw,
                    hours_per_week_raw,
                    native_country_raw,
                    income_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
                """,
                file
            )

    raw_conn.commit()

finally:
    raw_conn.close()

print("COPY completed.")

COPY completed.


In [10]:
pd.read_sql(
    """
    SELECT COUNT(*) AS loaded_rows
    FROM hw3_adult_staging;
    """,
    engine
)

,loaded_rows
0,48842


In [11]:
pd.read_sql(
    """
    SELECT
        age_raw,
        workclass_raw,
        education_raw,
        occupation_raw,
        hours_per_week_raw,
        native_country_raw,
        income_raw
    FROM hw3_adult_staging
    LIMIT 10;
    """,
    engine
)

,age_raw,workclass_raw,education_raw,occupation_raw,hours_per_week_raw,native_country_raw,income_raw
0,39,State-gov,Bachelors,Adm-clerical,40,United-States,<=50K
1,50,Self-emp-not-inc,Bachelors,Exec-managerial,13,United-States,<=50K
2,38,Private,HS-grad,Handlers-cleaners,40,United-States,<=50K
3,53,Private,11th,Handlers-cleaners,40,United-States,<=50K
4,28,Private,Bachelors,Prof-specialty,40,Cuba,<=50K
5,37,Private,Masters,Exec-managerial,40,United-States,<=50K
6,49,Private,9th,Other-service,16,Jamaica,<=50K
7,52,Self-emp-not-inc,HS-grad,Exec-managerial,45,United-States,>50K
8,31,Private,Masters,Prof-specialty,50,United-States,>50K
9,42,Private,Bachelors,Exec-managerial,40,United-States,>50K


In [12]:
pd.read_sql(
    """
    SELECT
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_name = 'hw3_adult_staging'
    ORDER BY ordinal_position;
    """,
    engine
)

,column_name,data_type
0,age_raw,text
1,workclass_raw,text
2,fnlwgt_raw,text
3,education_raw,text
4,education_num_raw,text
5,marital_status_raw,text
6,occupation_raw,text
7,relationship_raw,text
8,race_raw,text
9,sex_raw,text


Staging and DDL

In [13]:
clean_table_sql = """
DROP TABLE IF EXISTS hw3_adult CASCADE;

CREATE TABLE hw3_adult (
    person_id BIGINT
        GENERATED BY DEFAULT AS IDENTITY
        PRIMARY KEY,

    age SMALLINT NOT NULL
        CHECK (age BETWEEN 17 AND 100),

    workclass TEXT,

    fnlwgt INTEGER NOT NULL
        CHECK (fnlwgt > 0),

    education TEXT NOT NULL,

    education_num SMALLINT NOT NULL
        CHECK (education_num BETWEEN 1 AND 20),

    marital_status TEXT NOT NULL,

    occupation TEXT,

    relationship TEXT NOT NULL,

    race TEXT NOT NULL,

    sex TEXT NOT NULL,

    capital_gain NUMERIC(12,2) NOT NULL
        CHECK (capital_gain >= 0),

    capital_loss NUMERIC(12,2) NOT NULL
        CHECK (capital_loss >= 0),

    hours_per_week SMALLINT NOT NULL
        CHECK (hours_per_week BETWEEN 1 AND 168),

    native_country TEXT,

    income TEXT NOT NULL
        CHECK (income IN ('<=50K', '>50K'))
);
"""

with engine.begin() as conn:
    conn.execute(text(clean_table_sql))

print("Final table created.")

Final table created.


In [14]:
insert_sql = """
WITH parsed AS (
    SELECT
        NULLIF(TRIM(age_raw), '')::SMALLINT AS age,

        NULLIF(NULLIF(TRIM(workclass_raw), ''), '?')
            AS workclass,

        NULLIF(TRIM(fnlwgt_raw), '')::INTEGER
            AS fnlwgt,

        NULLIF(TRIM(education_raw), '')
            AS education,

        NULLIF(TRIM(education_num_raw), '')::SMALLINT
            AS education_num,

        NULLIF(TRIM(marital_status_raw), '')
            AS marital_status,

        NULLIF(NULLIF(TRIM(occupation_raw), ''), '?')
            AS occupation,

        NULLIF(TRIM(relationship_raw), '')
            AS relationship,

        NULLIF(TRIM(race_raw), '')
            AS race,

        NULLIF(TRIM(sex_raw), '')
            AS sex,

        NULLIF(TRIM(capital_gain_raw), '')::NUMERIC(12,2)
            AS capital_gain,

        NULLIF(TRIM(capital_loss_raw), '')::NUMERIC(12,2)
            AS capital_loss,

        NULLIF(TRIM(hours_per_week_raw), '')::SMALLINT
            AS hours_per_week,

        NULLIF(NULLIF(TRIM(native_country_raw), ''), '?')
            AS native_country,

        REPLACE(
            NULLIF(TRIM(income_raw), ''),
            '.',
            ''
        ) AS income

    FROM hw3_adult_staging
)

INSERT INTO hw3_adult (
    age,
    workclass,
    fnlwgt,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income
)

SELECT
    age,
    workclass,
    fnlwgt,
    education,
    education_num,
    marital_status,
    occupation,
    relationship,
    race,
    sex,
    capital_gain,
    capital_loss,
    hours_per_week,
    native_country,
    income

FROM parsed

WHERE age BETWEEN 17 AND 100
  AND fnlwgt > 0
  AND education IS NOT NULL
  AND education_num BETWEEN 1 AND 20
  AND marital_status IS NOT NULL
  AND relationship IS NOT NULL
  AND race IS NOT NULL
  AND sex IS NOT NULL
  AND capital_gain >= 0
  AND capital_loss >= 0
  AND hours_per_week BETWEEN 1 AND 168
  AND income IN ('<=50K', '>50K');
"""

with engine.begin() as conn:
    conn.execute(text(insert_sql))

print("Clean data inserted.")

Clean data inserted.


In [15]:
pd.read_sql(
    """
    SELECT
        person_id,
        age,
        workclass,
        education,
        occupation,
        hours_per_week,
        native_country,
        income
    FROM hw3_adult
    ORDER BY person_id
    LIMIT 10;
    """,
    engine
)

,person_id,age,workclass,education,occupation,hours_per_week,native_country,income
0,1,39,State-gov,Bachelors,Adm-clerical,40,United-States,<=50K
1,2,50,Self-emp-not-inc,Bachelors,Exec-managerial,13,United-States,<=50K
2,3,38,Private,HS-grad,Handlers-cleaners,40,United-States,<=50K
3,4,53,Private,11th,Handlers-cleaners,40,United-States,<=50K
4,5,28,Private,Bachelors,Prof-specialty,40,Cuba,<=50K
5,6,37,Private,Masters,Exec-managerial,40,United-States,<=50K
6,7,49,Private,9th,Other-service,16,Jamaica,<=50K
7,8,52,Self-emp-not-inc,HS-grad,Exec-managerial,45,United-States,>50K
8,9,31,Private,Masters,Prof-specialty,50,United-States,>50K
9,10,42,Private,Bachelors,Exec-managerial,40,United-States,>50K


In [16]:
pd.read_sql(
    """
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_name = 'hw3_adult'
    ORDER BY ordinal_position;
    """,
    engine
)

,column_name,data_type,is_nullable
0,person_id,bigint,NO
1,age,smallint,NO
2,workclass,text,YES
3,fnlwgt,integer,NO
4,education,text,NO
5,education_num,smallint,NO
6,marital_status,text,NO
7,occupation,text,YES
8,relationship,text,NO
9,race,text,NO


In [17]:
pd.read_sql(
    """
    SELECT 'staging' AS layer, COUNT(*) AS n_rows
    FROM hw3_adult_staging

    UNION ALL

    SELECT 'clean' AS layer, COUNT(*) AS n_rows
    FROM hw3_adult;
    """,
    engine
)

,layer,n_rows
0,staging,48842
1,clean,48842


Data quality audit

In [18]:
duplicates = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT (
            age,
            workclass,
            fnlwgt,
            education,
            education_num,
            marital_status,
            occupation,
            relationship,
            race,
            sex,
            capital_gain,
            capital_loss,
            hours_per_week,
            native_country,
            income
        )) AS distinct_records,

        COUNT(*) -
        COUNT(DISTINCT (
            age,
            workclass,
            fnlwgt,
            education,
            education_num,
            marital_status,
            occupation,
            relationship,
            race,
            sex,
            capital_gain,
            capital_loss,
            hours_per_week,
            native_country,
            income
        )) AS potential_duplicates

    FROM hw3_adult;
    """,
    engine
)

duplicates

,total_rows,distinct_records,potential_duplicates
0,48842,48790,52


**Висновок:** Перевірка показує можливі дублікати в даних. Оскільки в Adult немає унікального business key, це лише потенційні дублікати.

In [19]:
special_markers = pd.read_sql(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE TRIM(workclass_raw) = '?'
        ) AS missing_workclass,

        COUNT(*) FILTER (
            WHERE TRIM(occupation_raw) = '?'
        ) AS missing_occupation,

        COUNT(*) FILTER (
            WHERE TRIM(native_country_raw) = '?'
        ) AS missing_native_country,

        COUNT(*) FILTER (
            WHERE TRIM(income_raw) = '?'
        ) AS missing_income,

        COUNT(*) AS total_rows

    FROM hw3_adult_staging;
    """,
    engine
)

special_markers

,missing_workclass,missing_occupation,missing_native_country,missing_income,total_rows
0,1836,1843,583,0,48842


**Висновок:** У сирих даних `?` використовується як позначення пропущених значень. Під час очищення ці значення були замінені на `NULL`.

In [20]:
null_rates = pd.read_sql(
    """
    SELECT
        ROUND(
            COUNT(*) FILTER (WHERE workclass IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_workclass,

        ROUND(
            COUNT(*) FILTER (WHERE occupation IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_occupation,

        ROUND(
            COUNT(*) FILTER (WHERE native_country IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_native_country,

        ROUND(
            COUNT(*) FILTER (WHERE education IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_education,

        ROUND(
            COUNT(*) FILTER (WHERE hours_per_week IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_hours,

        ROUND(
            COUNT(*) FILTER (WHERE income IS NULL) * 100.0
            / NULLIF(COUNT(*), 0),
            2
        ) AS pct_null_income,

        COUNT(*) AS total_rows

    FROM hw3_adult;
    """,
    engine
)

null_rates

,pct_null_workclass,pct_null_occupation,pct_null_native_country,pct_null_education,pct_null_hours,pct_null_income,total_rows
0,5.73,5.75,1.75,0.0,0.0,0.0,48842


Пропущені значення найбільше зустрічаються в workclass (5.73%), occupation (5.75%) та native_country (1.75%). В інших перевірених колонках пропущених значень немає.

In [21]:
pd.read_sql(
    """
    SELECT
        person_id,
        age,
        workclass,
        occupation,
        native_country,
        income
    FROM hw3_adult
    WHERE workclass IS NULL
       OR occupation IS NULL
       OR native_country IS NULL
    ORDER BY person_id
    LIMIT 10;
    """,
    engine
)

,person_id,age,workclass,occupation,native_country,income
0,15,40,Private,Craft-repair,None,>50K
1,28,54,None,None,South,>50K
2,39,31,Private,Sales,None,>50K
3,52,18,Private,Other-service,None,<=50K
4,62,32,None,None,None,<=50K
5,70,25,None,None,United-States,<=50K
6,78,67,None,None,United-States,<=50K
7,94,30,Private,Sales,None,<=50K
8,107,17,None,None,United-States,<=50K
9,129,35,None,None,United-States,<=50K


Distributions and potential anomalies

In [22]:
income_distribution = pd.read_sql(
    """
    SELECT
        income,
        COUNT(*) AS n_rows,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_rows
    FROM hw3_adult
    GROUP BY income
    ORDER BY n_rows DESC, income;
    """,
    engine
)

income_distribution

,income,n_rows,pct_rows
0,<=50K,37155,76.07
1,>50K,11687,23.93


**Висновок:** Категорія `<=50K` зустрічається частіше, ніж `>50K`. Отже, розподіл доходу нерівномірний.

In [23]:
hours_stats = pd.read_sql(
    """
    SELECT
        COUNT(hours_per_week) AS n_non_null,
        MIN(hours_per_week) AS min_hours,
        MAX(hours_per_week) AS max_hours,
        ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_hours,
        ROUND(STDDEV_SAMP(hours_per_week)::NUMERIC, 2) AS sd_hours
    FROM hw3_adult;
    """,
    engine
)

hours_stats

,n_non_null,min_hours,max_hours,avg_hours,sd_hours
0,48842,1,99,40.42,12.39


**Висновок:** Кількість робочих годин має досить широкий діапазон. Середнє значення показує типовий рівень робочого навантаження в датасеті.

Outlier check

In [24]:
hours_outliers = pd.read_sql(
    """
    SELECT
        person_id,
        age,
        occupation,
        workclass,
        hours_per_week,
        income
    FROM hw3_adult
    WHERE hours_per_week > 80
    ORDER BY hours_per_week DESC, person_id
    LIMIT 20;
    """,
    engine
)

hours_outliers

,person_id,age,occupation,workclass,hours_per_week,income
0,936,37,Craft-repair,Private,99,>50K
1,1173,25,Farming-fishing,Private,99,>50K
2,1890,55,Farming-fishing,Self-emp-not-inc,99,>50K
3,3583,37,Prof-specialty,Self-emp-inc,99,>50K
4,4091,50,None,None,99,<=50K
5,4313,35,Farming-fishing,Self-emp-not-inc,99,<=50K
6,4446,43,Prof-specialty,Self-emp-inc,99,>50K
7,5380,43,Craft-repair,Private,99,<=50K
8,5436,44,Prof-specialty,Private,99,<=50K
9,6622,27,Transport-moving,Private,99,<=50K


**Висновок:** Значення понад 80 робочих годин на тиждень є незвичайними, але можливими. Я б залишила ці записи, але позначила їх як потенційні викиди.

DQL / EDA queries

**Бізнес-питання:** Який розподіл категорій доходу в датасеті?

In [25]:
pd.read_sql(
    """
    SELECT
        income,
        COUNT(*) AS n_rows,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
    FROM hw3_adult
    GROUP BY income
    ORDER BY n_rows DESC;
    """,
    engine
)

,income,n_rows,pct_rows
0,<=50K,37155,76.07
1,>50K,11687,23.93


**Висновок:** Більшість людей мають дохід `<=50K`. Категорія `>50K` зустрічається рідше.

**Бізнес-питання:** Які категорії освіти мають найбільшу частку людей з доходом понад 50K?

In [26]:
pd.read_sql(
    """
    SELECT
        education,
        COUNT(*) AS total_people,
        COUNT(*) FILTER (WHERE income = '>50K') AS high_income,
        ROUND(
            COUNT(*) FILTER (WHERE income = '>50K') * 100.0
            / COUNT(*),
            2
        ) AS pct_high_income
    FROM hw3_adult
    GROUP BY education
    ORDER BY pct_high_income DESC
    LIMIT 10;
    """,
    engine
)

,education,total_people,high_income,pct_high_income
0,Prof-school,834,617,73.98
1,Doctorate,594,431,72.56
2,Masters,2657,1459,54.91
3,Bachelors,8025,3313,41.28
4,Assoc-acdm,1601,413,25.80
5,Assoc-voc,2061,522,25.33
6,Some-college,10878,2063,18.96
7,HS-grad,15784,2503,15.86
8,12th,657,48,7.31
9,7th-8th,955,62,6.49


**Висновок:** Частка людей з доходом `>50K` відрізняється залежно від освіти. У деяких категоріях освіти вона значно вища.

**Бізнес-питання:** Як відрізняється дохід залежно від кількості робочих годин на тиждень?

In [27]:
pd.read_sql(
    """
    SELECT
        CASE
            WHEN hours_per_week BETWEEN 1 AND 30 THEN '1-30'
            WHEN hours_per_week BETWEEN 31 AND 40 THEN '31-40'
            WHEN hours_per_week BETWEEN 41 AND 50 THEN '41-50'
            ELSE '51+'
        END AS hours_group,

        COUNT(*) AS total_people,

        COUNT(*) FILTER (
            WHERE income = '>50K'
        ) AS high_income

    FROM hw3_adult
    GROUP BY hours_group
    ORDER BY hours_group;
    """,
    engine
)

,hours_group,total_people,high_income
0,1-30,7851,526
1,31-40,26639,5419
2,41-50,8917,3519
3,51+,5435,2223


**Висновок:** Кількість людей з доходом `>50K` відрізняється між групами робочих годин. Робочі години можуть бути корисними для аналізу доходу.

**Бізнес-питання:** Скільки записів не мають інформації про професію?

In [28]:
pd.read_sql(
    """
    SELECT COUNT(*) AS missing_occupation
    FROM hw3_adult
    WHERE occupation IS NULL;
    """,
    engine
)

,missing_occupation
0,2809


**Висновок:** У датасеті є записи без інформації про професію. Після очищення вони зберігаються як `NULL`.

**Бізнес-питання:** Які п’ять професій найчастіше зустрічаються в датасеті?

In [29]:
pd.read_sql(
    """
    SELECT
        occupation,
        COUNT(*) AS n_people
    FROM hw3_adult
    WHERE occupation IS NOT NULL
    GROUP BY occupation
    ORDER BY n_people DESC, occupation
    LIMIT 5;
    """,
    engine
)

,occupation,n_people
0,Prof-specialty,6172
1,Craft-repair,6112
2,Exec-managerial,6086
3,Adm-clerical,5611
4,Sales,5504


**Висновок:** Деякі професії зустрічаються частіше за інші. Розподіл професій у датасеті нерівномірний.

**Бізнес-питання:** Які унікальні країни, назви яких починаються на C, є в датасеті?

In [30]:
pd.read_sql(
    """
    SELECT DISTINCT
        native_country
    FROM hw3_adult
    WHERE native_country ILIKE 'C%%'
    ORDER BY native_country;
    """,
    engine
)

,native_country
0,Cambodia
1,Canada
2,China
3,Columbia
4,Cuba


**Висновок:** Запит показав унікальні країни, назви яких починаються на `C`. Кожна країна показана лише один раз.

**Бізнес-питання:** Скільки людей віком 30–50 років мають дохід понад 50K, належать до вибраних категорій зайнятості та працюють не менше 20 годин на тиждень?

In [31]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS n_people
    FROM hw3_adult
    WHERE age BETWEEN 30 AND 50
      AND income = '>50K'
      AND (
            workclass IN ('Private', 'Self-emp-not-inc', 'Self-emp-inc')
            OR occupation = 'Exec-managerial'
          )
      AND NOT (hours_per_week < 20);
    """,
    engine
)

,n_people
0,6592


**Висновок:** Запит показує кількість людей, які відповідають усім заданим умовам. Тут одночасно враховуються вік, дохід, зайнятість і робочі години.

**Бізнес-питання:** Скільки записів належить до кожної категорії `workclass`, якщо пропущені значення показати як `Unknown`?

In [32]:
pd.read_sql(
    """
    SELECT
        COALESCE(workclass, 'Unknown') AS workclass_group,
        COUNT(*) AS n_people
    FROM hw3_adult
    GROUP BY COALESCE(workclass, 'Unknown')
    ORDER BY n_people DESC;
    """,
    engine
)

,workclass_group,n_people
0,Private,33906
1,Self-emp-not-inc,3862
2,Local-gov,3136
3,Unknown,2799
4,State-gov,1981
5,Self-emp-inc,1695
6,Federal-gov,1432
7,Without-pay,21
8,Never-worked,10


**Висновок:** Запит показує розподіл людей за категоріями `workclass`. Пропущені значення окремо показані як `Unknown`.

**Бізнес-питання:** Скільки людей мають відому професію, але їхній `workclass` відрізняється від `Private`?

In [33]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS n_people
    FROM hw3_adult
    WHERE occupation IS NOT NULL
      AND workclass IS DISTINCT FROM 'Private';
    """,
    engine
)

,n_people
0,12127


**Висновок:** У датасеті є люди з відомою професією, які не належать до категорії `Private`. Запит також правильно враховує можливі `NULL`.

**Бізнес-питання:** Які записи мають найбільше значення `capital_gain`?

In [34]:
pd.read_sql(
    """
    SELECT
        person_id,
        age,
        education,
        occupation,
        capital_gain,
        income
    FROM hw3_adult
    WHERE capital_gain > 0
    ORDER BY capital_gain DESC
    LIMIT 10;
    """,
    engine
)

,person_id,age,education,occupation,capital_gain,income
0,1829,22,HS-grad,Prof-specialty,99999.0,>50K
1,1774,50,Some-college,Exec-managerial,99999.0,>50K
2,1768,49,Bachelors,Prof-specialty,99999.0,>50K
3,1370,52,HS-grad,Exec-managerial,99999.0,>50K
4,1485,53,HS-grad,Sales,99999.0,>50K
5,1531,52,Bachelors,Exec-managerial,99999.0,>50K
6,1619,46,Prof-school,Prof-specialty,99999.0,>50K
7,1685,43,Bachelors,Exec-managerial,99999.0,>50K
8,1247,54,Prof-school,Prof-specialty,99999.0,>50K
9,2106,43,Some-college,Sales,99999.0,>50K


**Висновок:** Запит показує записи з найбільшими значеннями `capital_gain`. Ці значення можна додатково перевірити як потенційно незвичайні.

## 6. Reflection

Під час роботи з Adult dataset я побачила, що основна проблема — це пропущені дані. У сирих даних вони були позначені знаком `?`. Під час очищення я замінила їх на `NULL`. Найбільше пропусків було в `occupation` (5.75%) і `workclass` (5.73%). У `native_country` пропусків було 1.75%. Також я перевірила дані на дублікати та незвичайні числові значення.

Для ML я б використала `age`, `education`, `workclass`, `occupation`, `hours_per_week`, `capital_gain`, `capital_loss` та `native_country`. Ці колонки можуть бути корисними для прогнозування доходу. `person_id` я б не використовувала, тому що це просто номер рядка, який ми створили самі.

`income` також не можна використовувати як feature, тому що це значення, яке ми хочемо прогнозувати. У цьому датасеті задача ML — classification, тому що потрібно визначити одну з двох категорій: `>50K` або `<=50K`.

Я не знайшла очевидної проблеми з data leakage. Але перед створенням моделі все одно потрібно перевірити, чи всі вибрані дані доступні на момент прогнозу.

Для покращення pipeline я б автоматично замінювала `?` на `NULL`, перевіряла правильність числових значень і дублікати. Також корисно автоматично перевіряти кількість рядків після кожного завантаження. Це допоможе швидше знаходити проблеми в нових даних.